In [ ]:
import subprocess
import numpy as np
import platform
import os
from pathlib import Path
from typing import List, Tuple, Optional
import time
import shutil


class A5_GMR_2:

    def __init__(self, executable_path: Optional[str] = None, timeout: int = 30):
        self.timeout = timeout
        self.system = platform.system()
        
        if executable_path is None:
            self.executable_path = self._get_default_executable_path()
        else:
            self.executable_path = executable_path
        
        print(f"System: {self.system}")
        print(f"Executable: {self.executable_path}")
    
    def _get_default_executable_path(self) -> str:
        if self.system == "Linux" or self.system == "Darwin":
            return os.path.join(".", "LINUX", "a5_gmr2_attack")
        elif self.system == "Windows":
            return os.path.join(".", "Windows", "A5-GMR-2-Attack", "x64", "Release", "A5-GMR-2-Attack_WINDOWS.exe")
        else:
            return os.path.join(".", "LINUX", "a5_gmr2_attack")
    
    def _run_program(self, num_keystream: int = 2) -> Tuple[Optional[List[int]], Optional[List[int]]]:
        if not 1 <= num_keystream <= 5:
            raise ValueError("num_keystream must be between 1 and 5")
        
        try:
            cmd = [self.executable_path, str(num_keystream)]
            print(f"Executing: {' '.join(cmd)}")
            
            result = subprocess.run(cmd, capture_output=True, text=True, 
                                  timeout=self.timeout, cwd=os.getcwd())
            
            if result.returncode != 0:
                print(f"Program execution error (return code {result.returncode}):")
                print(f"STDERR: {result.stderr}")
                return None, None
            
            lines = result.stdout.strip().split('\n')
            
            # Filter numeric lines
            numeric_lines = []
            for line in lines:
                line = line.strip()
                try:
                    list(map(int, line.split()))
                    numeric_lines.append(line)
                except ValueError:
                    continue
            
            if len(numeric_lines) < 2:
                print(f"Insufficient numeric output lines. Got: {numeric_lines}")
                return None, None
                
            first_line = list(map(int, numeric_lines[0].split()))
            second_line = list(map(int, numeric_lines[1].split()))
            
            return first_line, second_line
            
        except subprocess.TimeoutExpired:
            print(f"Program execution timeout ({self.timeout}s)")
            return None, None
        except FileNotFoundError:
            print(f"Executable not found: {self.executable_path}")
            return None, None
        except Exception as e:
            print(f"Output parsing error: {e}")
            return None, None
    
    def load_e_matrix(self, filename: str = "e_matrix.txt") -> Optional[np.ndarray]:
        try:
            if not Path(filename).exists():
                print(f"File '{filename}' not found.")
                return None
            
            e_matrix = np.loadtxt(filename, dtype=int)
            print(f"Matrix loaded: {e_matrix.shape}")
            return e_matrix
            
        except Exception as e:
            print(f"Error loading e_matrix: {e}")
            return None
    
    def run_dwave_attack(self, num_keystream: int = 2, 
                        dwave_solver: Optional[str] = None,
                        test_num: Optional[int] = None,
                        num_reads: int = 100) -> dict:
        """Standard D-Wave attack: single e_matrix"""
        
        if test_num is None:
            test_num = self._generate_test_number()
        
        run_dir = self._create_run_directory(test_num)
        
        print("Starting D-Wave Attack...")
        print(f"Test Number: {test_num}")
        print("=" * 50)
        
        try:
            # Phase 1: Generate e_matrix
            print("Phase 1: Generating e_matrix...")
            first, second = self._run_program(num_keystream)
            if first is None or second is None:
                return {"error": "E_matrix generation failed", "test_num": test_num}
            
            e_matrix = self.load_e_matrix("e_matrix.txt")
            if e_matrix is None:
                return {"error": "Failed to load e_matrix", "test_num": test_num}
            
            # Phase 2: Convert to QUBO and run D-Wave
            print("Phase 2: Converting to QUBO and running D-Wave...")
            Q_matrix = self._create_standard_qubo(e_matrix)
            quantum_results = self._run_dwave_computation(Q_matrix, dwave_solver, num_reads)
            
            if "error" in quantum_results:
                return {"error": f"D-Wave computation failed: {quantum_results['error']}", "test_num": test_num}
            
            # Phase 3: Analyze results
            print("Phase 3: Analyzing attack success...")
            attack_analysis = self._analyze_attack_success([(first, second)], quantum_results)
            
            # Save results
            self._save_results(run_dir, {
                "test_num": test_num,
                "executable_keys": (first, second),
                "quantum_results": quantum_results,
                "attack_analysis": attack_analysis,
                "attack_success": attack_analysis["success"]
            })
            
            # Final status
            if attack_analysis["success"]:
                print(f"SUCCESS! Found {len(attack_analysis['matches'])} matching key(s)")
            else:
                print("Attack failed - no matching keys found")
            
            return {
                "test_num": test_num,
                "run_dir": run_dir,
                "attack_success": attack_analysis["success"],
                "quantum_results": quantum_results,
                "attack_analysis": attack_analysis
            }
            
        except Exception as e:
            error_msg = f"D-Wave attack failed: {str(e)}"
            print(f"ERROR: {error_msg}")
            return {"error": error_msg, "test_num": test_num}
    
    def _create_standard_qubo(self, e_matrix: np.ndarray) -> np.ndarray:
        """Create QUBO matrix from e_matrix"""
        n_vars = e_matrix.shape[0]
        n = 8  # Number of groups
        m = n_vars // n  # Variables per group
        penalty = 1.1
        
        Q = np.zeros((n_vars, n_vars))
        
        for i in range(n_vars):
            for j in range(n_vars):
                if i == j:
                    Q[i][j] -= 1  # Diagonal: -1
                elif i // m == j // m:
                    Q[i][j] += penalty  # Same group: +penalty
                
                if e_matrix[i][j] == 1:  # Adjacent relationship
                    Q[i][i] += penalty  # Add penalty to diagonal
                    Q[i][j] -= penalty  # Subtract penalty from interaction
        
        # Convert to upper triangular form
        for i in range(n_vars):
            for j in range(n_vars):
                if i != j and j > i:
                    Q[i][j] += Q[j][i]
                    Q[j][i] = 0
        
        return Q
    
    def _run_dwave_computation(self, qubo_matrix: np.ndarray, solver: Optional[str] = None,
                              num_reads: int = 100) -> dict:
        """Run D-Wave quantum computation on QUBO matrix"""
        try:
            from dimod import BinaryQuadraticModel
            from dwave.system import LeapHybridSampler, DWaveSampler, EmbeddingComposite
            from dwave.samplers import SimulatedAnnealingSampler
            import dimod
        except ImportError:
            return {"error": "D-Wave libraries not installed"}
        
        solver_name = solver if solver else "hybrid_binary_quadratic_model_version2p"
        is_hybrid = ("hybrid" in solver_name.lower()) or (solver is None)
        
        print(f"QUBO matrix shape: {qubo_matrix.shape}")
        
        # Convert to QUBO dictionary
        QQ = {}
        for i in range(qubo_matrix.shape[0]):
            for j in range(qubo_matrix.shape[1]):  
                if qubo_matrix[i, j] != 0:
                    QQ[(i, j)] = qubo_matrix[i, j]
        
        # Create BQM
        bqm = dimod.BinaryQuadraticModel.from_qubo(QQ)
        print(f"BQM created with {len(bqm.variables)} variables")
        
        # Set up sampler
        try:
            if solver_name == "simulated":
                sampler = SimulatedAnnealingSampler()
                actual_solver_name = "simulated_annealing"
            elif is_hybrid:
                sampler = LeapHybridSampler()
                actual_solver_name = solver_name
            else:
                base_sampler = DWaveSampler(solver=solver_name)
                sampler = EmbeddingComposite(base_sampler)
                actual_solver_name = solver_name
        except:
            sampler = SimulatedAnnealingSampler()
            actual_solver_name = "simulated_annealing"
        
        # Run computation
        print(f"Submitting to {actual_solver_name}...")
        start_time = time.time()
        
        if "simulated" in actual_solver_name:
            sampleset = sampler.sample(bqm, num_reads=num_reads)
        elif is_hybrid:
            sampleset = sampler.sample(bqm, label="A5_GMR_2_Attack")
        else:
            sampleset = sampler.sample(bqm, num_reads=num_reads)
        
        computation_time = time.time() - start_time
        print(f"Computation completed in {computation_time:.2f}s")
        
        # Process results
        rec = sampleset.first
        sample, energy = rec.sample, rec.energy
        selected_indices = [i for i, v in sample.items() if v == 1]
        
        print(f"[solver] {actual_solver_name}")
        print(f"energy: {energy}")
        print(f"selected indices: {selected_indices}")
        
        # Extract timing info
        timing_info = {}
        if hasattr(sampleset, 'info') and sampleset.info:
            info = sampleset.info[0] if isinstance(sampleset.info, list) else sampleset.info
            timing_info = {
                'qpu_access_time': info.get('qpu_access_time', None),
                'qpu_programming_time': info.get('qpu_programming_time', None),
                'qpu_anneal_time_per_sample': info.get('qpu_anneal_time_per_sample', None),
                'charge_time': info.get('charge_time', None),
                'run_time': info.get('run_time', None),
            }
            
            # Print timing info
            for key, value in timing_info.items():
                if value is not None:
                    print(f"{key}: {value/1000:.3f} ms")
        
        return {
            "success": True,
            "solver": actual_solver_name,
            "computation_time": computation_time,
            "best_energy": float(energy),
            "best_sample": sample,
            "selected_indices": selected_indices,
            "dwave_keys": [selected_indices] if selected_indices else [],
            "timing_info": timing_info
        }
    
    def _analyze_attack_success(self, executable_keys: List[Tuple[List[int], List[int]]], 
                               quantum_results: dict) -> dict:
        """Compare executable keys with D-Wave results"""
        if "best_sample" not in quantum_results:
            return {"success": False, "error": "No quantum solution found", "matches": []}
        
        dwave_keys = quantum_results.get("dwave_keys", [])
        matches = []
        
        for first, second in executable_keys:
            for dwave_key in dwave_keys:
                if set(first) == set(dwave_key):
                    matches.append({
                        "type": "first_key",
                        "executable_key": first, "dwave_key": dwave_key
                    })
                elif set(second) == set(dwave_key):
                    matches.append({
                        "type": "second_key",
                        "executable_key": second, "dwave_key": dwave_key
                    })
        
        return {
            "success": len(matches) > 0,
            "matches": matches,
            "match_count": len(matches)
        }
    
    def _generate_test_number(self) -> int:
        run_data_dir = Path("run_data")
        if not run_data_dir.exists():
            return 1
        
        existing_tests = []
        for item in run_data_dir.iterdir():
            if item.is_dir() and item.name.startswith("test_"):
                try:
                    test_num = int(item.name.split("_")[1])
                    existing_tests.append(test_num)
                except ValueError:
                    continue
        
        return max(existing_tests, default=0) + 1
    
    def _create_run_directory(self, test_num: int) -> str:
        run_data_dir = Path("run_data")
        run_data_dir.mkdir(exist_ok=True)
        
        test_dir = run_data_dir / f"test_{test_num:03d}"
        test_dir.mkdir(exist_ok=True)
        
        return str(test_dir)
    
    def _save_results(self, run_dir: str, results: dict):
        """Save results to files"""
        results_file = os.path.join(run_dir, "results.txt")
        with open(results_file, 'w') as f:
            f.write(f"A5_GMR_2 Attack Results\n")
            f.write(f"======================\n")
            f.write(f"Test Number: {results['test_num']}\n")
            f.write(f"Attack Success: {results['attack_success']}\n\n")
            
            f.write(f"Executable Keys:\n")
            f.write(f"  First:  {results['executable_keys'][0]}\n")
            f.write(f"  Second: {results['executable_keys'][1]}\n\n")
            
            quantum = results['quantum_results']
            f.write(f"Quantum Results:\n")
            f.write(f"  Solver: {quantum.get('solver', 'Unknown')}\n")
            f.write(f"  Energy: {quantum.get('best_energy', 'N/A')}\n")
            f.write(f"  Computation Time: {quantum.get('computation_time', 0):.2f}s\n\n")
            
            dwave_keys = quantum.get('dwave_keys', [])
            f.write(f"D-Wave Keys ({len(dwave_keys)} found):\n")
            for i, key in enumerate(dwave_keys, 1):
                f.write(f"  Key {i}: {key}\n")

    def run_repeated_attacks(self, num_iterations: int = 100, 
                            num_keystream: int = 3,
                            dwave_solver: Optional[str] = None,
                            num_reads: int = 100) -> dict:
        """Execute D-Wave attacks repeatedly and collect statistics"""

        print(f"Running {num_iterations} D-Wave attacks...")
        print(f"Parameters: keystream={num_keystream}, solver={dwave_solver}")
        print("=" * 50)

        # Create directory for this batch
        from datetime import datetime
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        batch_dir = Path("repeated_execution_results") / f"batch_{num_iterations}x_{timestamp}"
        batch_dir.mkdir(parents=True, exist_ok=True)

        success_count = 0
        qpu_times = []
        computation_times = []
        energies = []

        for i in range(num_iterations):
            if (i + 1) % 10 == 0:  # Progress every 10 iterations
                print(f"Progress: {i + 1}/{num_iterations}")

            # Run single attack
            result = self.run_dwave_attack(
                num_keystream=num_keystream,
                dwave_solver=dwave_solver,
                num_reads=num_reads
            )

            # Check success
            if result.get('attack_success', False):
                success_count += 1

            # Save e_matrix for each iteration
            if os.path.exists("e_matrix.txt"):
                e_matrix_file = batch_dir / f"e_matrix_{i+1:03d}.txt"
                shutil.copy2("e_matrix.txt", e_matrix_file)

            # Collect timing data for statistics
            quantum = result.get('quantum_results', {})
            if quantum:
                if quantum.get('computation_time'):
                    computation_times.append(quantum['computation_time'])
                if quantum.get('best_energy'):
                    energies.append(quantum['best_energy'])
                
                timing = quantum.get('timing_info', {})
                qpu_access = timing.get('qpu_access_time')
                if qpu_access:
                    qpu_times.append(qpu_access / 1000)

        # Calculate statistics
        success_rate = (success_count / num_iterations) * 100

        stats = {
            'success_count': success_count,
            'success_rate': success_rate,
            'total_runs': num_iterations
        }

        if qpu_times:
            stats['qpu_access_time'] = {
                'mean': np.mean(qpu_times),
                'std': np.std(qpu_times),
                'min': np.min(qpu_times),
                'max': np.max(qpu_times)
            }

        if computation_times:
            stats['computation_time'] = {
                'mean': np.mean(computation_times),
                'std': np.std(computation_times)
            }

        # Print results
        print("\n" + "=" * 50)
        print("Repeated Execution Results")
        print("=" * 50)
        print(f"Success Rate: {success_rate:.1f}% ({success_count}/{num_iterations})")

        if qpu_times:
            qpu_stats = stats['qpu_access_time']
            print(f"QPU Access Time: {qpu_stats['mean']:.4f}s ± {qpu_stats['std']:.4f}s")
            print(f"QPU Range: {qpu_stats['min']:.4f}s - {qpu_stats['max']:.4f}s")

        print(f"All e_matrices saved to: {batch_dir}")

        return stats


In [ ]:

a5_gmr_2 = A5_GMR_2()
stats = a5_gmr_2.run_repeated_attacks(3, 3)